# Data Manipulation — Intermediate
## Reshaping & Combining Data (R)

**Philosophy:** Assumes clean data and familiarity with the tidyverse API. Each
section covers a multi-step pattern — not individual function syntax, but how
to chain tools together to solve a real reshaping problem.

---

## Decision Table

| If your problem is... | Go to |
| :--- | :--- |
| Data is wide and you need one row per observation | §1 — Wide vs Long |
| Need top-N per group, conditional aggregation, or filter-then-agg | §2 — Advanced GroupBy |
| Joining 3+ tables, row count explodes after merge, or need anti-join | §3 — Multi-table Pipelines |
| Need to change time frequency, align two time series, or forward-fill after resample | §4 — Time-Series Reshaping |
| Column contains JSON, lists, or list-columns | §5 — Nested & List-like Data |
| Reading and combining many CSV or Excel files | §6 — Combining Multiple Files |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` **(this note)** | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x. A small sample frame is defined below
> for runnable examples.

In [ ]:
library(dplyr)
library(tidyr)
library(lubridate)

df <- tibble::tibble(
  user_id    = c(1, 1, 2, 2, 3),
  region     = c('US','US','EU','EU','US'),
  platform   = c('iOS','Android','iOS','iOS','Web'),
  revenue    = c(100, 150, 200, 120, 90),
  event_date = ymd(c('2024-01-01','2024-02-01','2024-01-15','2024-03-01','2024-02-20'))
)
print(df)

---
## §1 — Wide vs Long Format

**Wide:** one row per entity, one column per time period or category. Easy to
read, hard to aggregate.
**Long:** one row per observation, categories stored as values in a column.
Required for most group-by/plotting/modeling operations.

In [ ]:
library(tidyr)
library(stringr)

# ── Wide -> Long: pivot_longer ───────────────────────────────────────────────
# Wide format (common from Excel or pivot exports):
# user_id | Q1_revenue | Q2_revenue | Q3_revenue
wide <- tibble::tibble(
  user_id    = c(1, 2, 3),
  Q1_revenue = c(100, 200, 150),
  Q2_revenue = c(120, 210, 160),
  Q3_revenue = c(130, 190, 170)
)

long <- wide |> pivot_longer(
  cols      = c(Q1_revenue, Q2_revenue, Q3_revenue),  # columns to unpivot
  names_to  = "quarter",                               # name for the new label column
  values_to = "revenue"                                # name for the new value column
)
# Result: user_id | quarter     | revenue
#         1       | Q1_revenue  | 100
#         1       | Q2_revenue  | 120  ...

# Clean the label column after pivoting -- can also be done inline via names_pattern, shown below
long <- long |> mutate(quarter = str_remove(quarter, "_revenue"))

# ── Long -> Wide: pivot_wider ────────────────────────────────────────────────
# Reverse the operation -- go back to one column per quarter
wide_again <- long |> group_by(user_id, quarter) |> summarise(revenue = sum(revenue), .groups = "drop") |>
  pivot_wider(id_cols = user_id, names_from = quarter, values_from = revenue)
# No "axis name" cleanup needed -- pivot_wider produces plain column names directly, unlike
# pandas' pivot_table which leaves a `columns.name` attribute behind

# ── When to use which format ────────────────────────────────────────────────
# Use LONG for:  group_by, plotting (ggplot2 wants long data almost universally), modeling
# Use WIDE for:  human-readable reports, correlation matrices, displaying to stakeholders

# ── Multiple value columns -- pivot_longer with names_pattern/names_sep ─────
# pandas' wide_to_long expects {stub}{sep}{suffix} columns; pivot_longer handles
# this directly via names_sep, splitting each column name into TWO new columns
# (.value selects which piece becomes the new column NAME vs becomes a value)
wide2 <- tibble::tibble(
  user_id = c(1, 2),
  rev_Q1 = c(100, 200), cost_Q1 = c(60, 110),
  rev_Q2 = c(120, 210), cost_Q2 = c(70, 120)
)
df_long <- wide2 |> pivot_longer(
  cols = -user_id,
  names_to = c(".value", "quarter"),   # .value = the stub becomes the new column name (rev / cost)
  names_sep = "_"
)
# df_long: user_id | quarter | rev | cost  (one row per user x quarter)

**Common mistakes:**
- Forgetting `id_cols=` in `pivot_wider()` when extra grouping columns are still present — without it, dplyr may try to use every remaining column as part of the row key, producing more rows than expected
- `pivot_wider()` raises (or produces list-columns) with duplicate index/column pairs unless `values_fn=` is supplied — pass `values_fn = sum` (or another aggregator) explicitly when duplicates are possible, the direct analog to pandas needing `pivot_table` instead of `pivot`
- Forgetting that `pivot_longer`'s `.value` sentinel in `names_to` is what makes the stub-splitting pattern work — omitting it just produces one combined label column, not separate `rev`/`cost` columns

**Verified — the `.value` / `names_sep` pattern, the R analog to `pd.wide_to_long`:**

In [ ]:
wide2 <- tibble::tibble(user_id = c(1, 2),
                        rev_Q1 = c(100, 200), cost_Q1 = c(60, 110),
                        rev_Q2 = c(120, 210), cost_Q2 = c(70, 120))
out <- wide2 |> pivot_longer(cols = -user_id, names_to = c(".value", "quarter"), names_sep = "_")
print(out)

---
## §2 — Advanced GroupBy Patterns

Beyond basic `group_by() |> summarise()` — patterns for filtering groups,
selecting top-N within a group, and conditional aggregation. Syntax reference
is in the Tidyverse reference (§6); this section focuses on the multi-step
patterns.

In [ ]:
library(dplyr)

# ── Pattern 1: Top-N per group ───────────────────────────────────────────────
# Goal: top 3 products by revenue within each region
# Method A: rank + filter (preserves all columns, most flexible)
df <- df |> group_by(region) |> mutate(rev_rank = dense_rank(desc(revenue))) |> ungroup()
top3 <- df |> filter(rev_rank <= 3) |> select(-rev_rank)

# Method B: arrange + group_by slice_head (simpler but loses ranks)
top3 <- df |> arrange(desc(revenue)) |> group_by(region) |> slice_head(n = 3) |> ungroup()

# ── Pattern 2: Filter groups by an aggregate condition ───────────────────────
# Goal: keep only regions where total revenue > 100,000
# group_by() |> filter() -- the predicate is evaluated PER GROUP automatically
# when it references an aggregate inside a grouped filter -- dplyr's equivalent
# of pandas' groupby().filter(lambda g: ...), but expressed without a lambda
active_regions <- df |> group_by(region) |> filter(sum(revenue) > 100000) |> ungroup()
# Result: same shape as df but entire groups removed if condition is False

# ── Pattern 3: Conditional aggregation ──────────────────────────────────────
# Goal: SUM(revenue WHERE platform = 'iOS') -- SQL conditional agg equivalent
df <- df |> mutate(
  ios_rev     = if_else(platform == "iOS",     revenue, 0),
  android_rev = if_else(platform == "Android", revenue, 0)
)

result <- df |> group_by(region) |> summarise(
  total_rev    = sum(revenue, na.rm = TRUE),
  ios_rev      = sum(ios_rev, na.rm = TRUE),
  android_rev  = sum(android_rev, na.rm = TRUE),
  unique_users = n_distinct(user_id),
  .groups = "drop"
)
result <- result |> mutate(ios_share = ios_rev / total_rev)

# ── Pattern 4: Group-level features back on original rows ───────────────────
# Goal: add group mean, count, and pct-of-total back to each row
df <- df |> group_by(region) |> mutate(
  region_avg_rev = mean(revenue, na.rm = TRUE),
  region_count   = n_distinct(user_id),
  pct_of_region  = revenue / sum(revenue, na.rm = TRUE)
) |> ungroup()

# ── Pattern 5: Multi-level groupby ──────────────────────────────────────────
# Goal: revenue per region x platform x month
df <- df |> mutate(year_month = format(event_date, "%Y-%m"))
result <- df |> group_by(region, platform, year_month) |> summarise(
  revenue = sum(revenue, na.rm = TRUE),
  users   = n_distinct(user_id),
  .groups = "drop"
)

# ── Pattern 6: Custom aggregation function ───────────────────────────────────
# Goal: 90th percentile revenue per region
result <- df |> group_by(region) |> summarise(
  p90_revenue = quantile(revenue, 0.9, na.rm = TRUE),
  cv_revenue  = sd(revenue, na.rm = TRUE) / mean(revenue, na.rm = TRUE),  # coefficient of variation
  .groups = "drop"
)

**Pattern selection:**

| Goal | Tool |
| :--- | :--- |
| Top-N per group | `group_by() |> mutate(dense_rank())` + filter, or `arrange + group_by() |> slice_head(n)` |
| Remove entire groups that fail a condition | `group_by() |> filter(aggregate_condition)` |
| Conditional sum/count within a group | `if_else()` to zero-out + `summarise()` |
| Group stats back on each row | `group_by() |> mutate()` |
| Custom aggregation (percentile, CV) | `summarise(stat = quantile(x, ...))` |

**Common mistakes:**
- Using `summarise()` when `mutate()` is needed — `summarise()` collapses rows and loses the join-back to original data, the same trap as pandas' `agg` vs `transform`
- Using `group_by() |> filter()` when you only want to flag — filter removes rows permanently; use `mutate()` to add a logical flag column instead
- Forgetting `ungroup()` after a grouped `mutate()` or `filter()` — the grouping silently persists into later pipe steps and corrupts subsequent aggregate computations

---
## §3 — Multi-table Pipelines

Single-join operations are in the Tidyverse reference (§5). This section covers
sequences of joins, diagnosing unexpected row count changes, and anti-join patterns.

In [ ]:
library(dplyr)

# ── Always check row counts at each join step ───────────────────────────────
join_check <- function(left, right, by, type = "left_join") {
  # Join and print row count before/after to detect fan-out.
  result <- do.call(type, list(left, right, by = by))
  cat(sprintf('%d -> %d rows  (%+d)\n', nrow(left), nrow(result), nrow(result) - nrow(left)))
  result
}

# ── Pattern 1: Sequential 3-table join ──────────────────────────────────────
# users -> orders -> order_items
result <- users |>                                       # base: one row per user
  left_join(orders,      by = "user_id") |>               # left: keep all users
  left_join(order_items, by = "order_id")                 # left: keep all orders
# If users have multiple orders and orders have multiple items,
# row count grows: users x avg_orders x avg_items_per_order

# ── Diagnosing row count explosion ──────────────────────────────────────────
# Step 1: check if the join key is unique in the right table
summary(orders |> count(user_id) |> pull(n))
# If max > 1, the join will multiply rows -- decide: aggregate right table first

# Step 2: aggregate the right table before joining (recommended for 1-to-many)
orders_agg <- orders |> group_by(user_id) |> summarise(
  order_count = n(),
  total_spend = sum(amount, na.rm = TRUE),
  last_order  = max(order_date),
  .groups = "drop"
)
result <- users |> left_join(orders_agg, by = "user_id")  # now 1-to-1

# ── Pattern 2: Anti-join -- rows in A with no match in B ─────────────────────
# SQL: SELECT * FROM A WHERE A.id NOT IN (SELECT id FROM B)
# dplyr has a PURPOSE-BUILT verb for this -- no merge(indicator=True) + filter dance needed
active_users <- users |> anti_join(churned_users |> select(user_id), by = "user_id")

# ── Pattern 3: Many-to-many join -- handle explicitly ────────────────────────
# Tags table: one user can have many tags, one tag can belong to many users
# Direct join -> row explosion; resolve by checking what you actually need

# Option A: aggregate tags into a list-column per user first
tags_agg <- tags |> group_by(user_id) |> summarise(tag_list = list(tag), .groups = "drop")
result <- users |> left_join(tags_agg, by = "user_id")

# Option B: keep only one tag per user (e.g. most recent)
tags_dedup <- tags |> arrange(created_at) |> distinct(user_id, .keep_all = TRUE)
result <- users |> left_join(tags_dedup |> select(user_id, tag), by = "user_id")

# ── Pattern 4: Bridge table join (many-to-many via junction table) ───────────
# users <-> user_roles <-> roles
result <- users |>
  inner_join(user_roles, by = "user_id") |>
  left_join(roles, by = "role_id")

**Row count change after join — diagnostic guide:**

| Row count change | Likely cause | Fix |
| :--- | :--- | :--- |
| Count stays same | 1-to-1 match ✅ | — |
| Count increases | Right table has multiple rows per key | Aggregate right table first |
| Count decreases | Inner join dropped non-matching rows | Switch to `left_join` if needed |
| Count drops to near zero | Join key type mismatch (numeric vs character) | Cast keys to same type |

**Common mistakes:**
- Joining without checking key uniqueness — row explosion is silent and hard to detect later
- Joining on columns with different types — `*_join()` raises a type-mismatch **error** in R (unlike pandas, which silently produces zero matches), so this failure mode is actually caught earlier, but the fix is the same: cast both keys to the same type first
- Using `inner_join()` when you need all rows from the left table — rows drop silently

**Two §3 additions: join cardinality checks and `coalesce()` across frames.**

The `join_check` helper above prints row counts to *detect* fan-out after the
fact; dplyr's `*_join()` functions accept a `relationship =` argument (dplyr
1.1+) that **guards** against it directly, raising an error the moment a join
would violate the cardinality you expect — the direct equivalent of pandas'
`merge(..., validate=...)`. `coalesce()` fills gaps in a primary vector from a
backup, position-aligned (R has no separate `combine_first()` — `coalesce()`
covers both the single-vector and whole-frame cases).

In [ ]:
library(dplyr)
users  <- tibble::tibble(user_id = c(1, 2, 3), name = c('a','b','c'))
orders <- tibble::tibble(user_id = c(1, 1, 2), amt = c(10, 20, 30))   # user 1 appears twice

result <- tryCatch(
  left_join(users, orders, by = "user_id", relationship = "one-to-one"),
  error = function(e) {
    cat('relationship="one-to-one" ->', conditionMessage(e), '\n')
    NULL
  }
)
cat('relationship="one-to-many" -> rows:',
    nrow(left_join(users, orders, by = "user_id", relationship = "one-to-many")), '\n')

# coalesce -- fill NAs in primary from backup, position-aligned (R's combine_first equivalent)
primary <- c(1, NA, 3)
backup  <- c(9, 2,  9)
cat('coalesce(primary, backup):', coalesce(primary, backup), '\n')

---
## §4 — Time-Series Reshaping

Covers changing temporal granularity, filling gaps in time series, and aligning
two series that run on different frequencies. R has no single `.resample()`
method tied to a `DatetimeIndex` the way pandas does — instead, you derive a
period column with `lubridate::floor_date()` and `group_by()` on it, which is
slightly more verbose but makes the aggregation step fully explicit.

In [ ]:
library(dplyr)
library(lubridate)
library(tidyr)

# Setup: daily transaction data
df <- df |> mutate(date = ymd(event_date)) |> arrange(date)

# ── Pattern 1: Downsample -- daily -> weekly / monthly ────────────────────────
monthly <- df |> mutate(period = floor_date(date, "month")) |> group_by(period) |> summarise(revenue = sum(revenue, na.rm = TRUE), .groups = "drop")
weekly  <- df |> mutate(period = floor_date(date, "week"))  |> group_by(period) |> summarise(revenue = mean(revenue, na.rm = TRUE), .groups = "drop")
# Common floor_date units: 'day', 'week', 'month', 'quarter', 'year', 'hour' --
# the direct R analog to pandas' 'D'/'W'/'ME'/'QE'/'YE'/'h' frequency aliases

# Multiple columns at different aggregations
monthly <- df |> mutate(period = floor_date(date, "month")) |> group_by(period) |> summarise(
  revenue  = sum(revenue, na.rm = TRUE),
  users    = n_distinct(user_id),
  sessions = n(),
  .groups = "drop"
)

# ── Pattern 2: Forward-fill gaps after resampling ──────────────────────────────
# Problem: resampling creates NA for periods with no data
# Use case: subscription status -- carry last known status forward
daily_status <- df |> mutate(period = floor_date(date, "day")) |>
  group_by(period) |> summarise(subscription_status = last(subscription_status), .groups = "drop") |>
  fill(subscription_status, .direction = "down")  # fill gaps with previous known status

# ── Pattern 3: Upsample -- monthly -> daily ───────────────────────────────────
# Go from lower to higher frequency (creates NA for new dates)
date_spine <- tibble::tibble(date = seq(min(monthly_series$date), max(monthly_series$date), by = "day"))
daily <- date_spine |> left_join(monthly_series, by = "date")              # insert daily rows, fill with NA
daily <- daily |> fill(value, .direction = "down")                          # forward fill from monthly value
daily <- daily |> mutate(value = zoo::na.approx(value, na.rm = FALSE))      # linear interpolation (needs the zoo package)

# ── Pattern 4: Align two time series with different frequencies ──────────────
# Example: daily sales + monthly macro indicator
daily_sales   <- df_daily |> select(date, revenue)
monthly_macro <- df_macro |> select(date, gdp_growth)

# Reindex monthly macro to daily frequency, then forward fill
aligned <- daily_sales |>
  left_join(monthly_macro, by = "date") |>
  arrange(date) |>
  fill(gdp_growth, .direction = "down")

# ── Pattern 5: Create a complete date spine ──────────────────────────────────
# Problem: some dates are missing entirely from the data
# Solution: build a full date range and left join
date_spine <- tibble::tibble(date = seq(min(df$date), max(df$date), by = "day"))
complete <- date_spine |> left_join(df, by = "date") |> mutate(revenue = replace_na(revenue, 0))  # missing days = 0 revenue

**Common mistakes:**
- Forward-filling before checking sort order — `fill()` fills in current row order; always `arrange()` by date first
- Reaching for a single `.resample()`-style call out of pandas habit — R's `floor_date() |> group_by() |> summarise()` pattern is the idiomatic replacement; there is no equivalent single function, but the explicit version makes the aggregation choice (`sum` vs `mean`) unmissable
- Downsampling with `mean()` when `sum()` is correct — e.g. daily revenue should be summed, not averaged, when going monthly, identical trap to pandas
- Building a date spine with `seq(..., by = "day")` but the original dates have time-of-day components — strip to date-only with `as_date()` first, the R equivalent of pandas' `.dt.normalize()`

---
## §5 — Nested & List-like Data

Common when working with API responses, JSON exports, or data where one cell
contains a list. R's native list-column support (any column can hold arbitrary
R objects, including other vectors) makes this somewhat more natural than
pandas' object-dtype columns, but flattening/exploding still needs the same
deliberate handling.

In [ ]:
library(dplyr)
library(tidyr)
library(jsonlite)

# ── Pattern 1: Flatten JSON / list column ────────────────────────────────────
# Data: df$metadata contains lists like list(source = 'organic', campaign = 'summer')
meta_df <- df$metadata |> purrr::map_dfr(as_tibble)    # expands each key to a column (purrr's bind-rows-of-tibbles idiom)
df <- df |> select(-metadata) |> bind_cols(meta_df)

# Nested JSON (multiple levels deep) -- jsonlite::flatten() handles this directly,
# analogous to pandas' json_normalize(sep=, max_level=)
flat <- jsonlite::fromJSON(records_json, flatten = TRUE)   # flatten=TRUE auto-joins nested keys with '.'

# ── Pattern 2: Unnest (explode) a list column ────────────────────────────────
# Data: df$tags contains list('python', 'pandas', 'sql')
# Goal: one row per tag
df_exploded <- df |> unnest(tags)
# Before: user_id=1, tags=list('python','pandas') -> 1 row
# After:  user_id=1, tags='python' AND user_id=1, tags='pandas' -> 2 rows
# Unlike pandas' .explode(), unnest() does NOT need a separate reset_index(drop=True)
# call -- tibbles never carry a meaningful row-index the way pandas DataFrames do

# Unnest multiple list columns of the same length simultaneously
df_exploded <- df |> unnest(c(tags, scores))

# ── Pattern 3: Parse stringified lists ──────────────────────────────────────
# Data: df$tags contains the literal string "['python', 'pandas']"
# R has NO direct equivalent of Python's ast.literal_eval (safe literal parsing) --
# the practical fix is to recognize the string as JSON-like and parse it as JSON,
# NOT to eval() the R string (R's eval(parse(text=...)) is the unsafe analog to
# Python's eval() and should be avoided for the same security reasons).
# If the source data is genuinely JSON (just stored as text), parse it directly:
df <- df |> mutate(tags = purrr::map(tags, jsonlite::fromJSON))
df_exploded <- df |> unnest(tags)
# If the source is Python repr syntax (single quotes, not valid JSON), normalize
# quoting first: stringr::str_replace_all(tags, "'", '"') before fromJSON() --
# this is a deliberate, visible parsing step, not a generic "eval this string" call

# ── Pattern 4: Expand a list column into separate columns ───────────────────
# Data: df$scores contains list(84, 91, 76) (fixed length)
scores_df <- df$scores |> purrr::map_dfr(~ as_tibble(setNames(as.list(.x), c("score_1","score_2","score_3"))))
df <- df |> select(-scores) |> bind_cols(scores_df)

# ── Pattern 5: Flatten nested JSON from an API response ─────────────────────
# API response: list of nested records
records <- jsonlite::fromJSON(api_response_text, simplifyDataFrame = FALSE)
df <- jsonlite::fromJSON(api_response_text, flatten = TRUE)   # flatten=TRUE is jsonlite's record_path-free analog;
                                                                 # for true record_path-style extraction (pulling one
                                                                 # nested list out and keeping top-level fields as meta),
                                                                 # do it in two steps: extract the nested list, then
                                                                 # bind_cols() the repeated top-level fields back on

**Common mistakes:**
- Reaching for `eval(parse(text = ...))` to "parse" a stringified structure — this is R's `eval()` equivalent and carries the same security risk as Python's `eval()`; if the string is JSON-like, parse it with `jsonlite::fromJSON()` instead, never by evaluating it as code
- Calling `jsonlite::fromJSON()` on a column with `NA` values without checking first — raises an error; filter or replace `NA` with an empty JSON object string (`"{}"`) first
- Forgetting that `unnest()` requires the column to actually be a **list-column** (`is.list(df$col)` is `TRUE`) — a plain character column holding text that merely *looks* like a list needs Pattern 3's parsing step first

---
## §6 — Combining Multiple Files

Common in real-world data pipelines: monthly data exports, partitioned data
dumps, or multi-sheet Excel files. The challenge is schema consistency across
files.

In [ ]:
library(dplyr)
library(fs)
library(purrr)
library(readr)

# ── Pattern 1: Read and combine many CSVs ────────────────────────────────────
# Simple -- same schema across all files
files <- dir_ls("data/monthly", glob = "*.csv")        # fs::dir_ls -- R's analog to glob.glob() / Path.glob()
df <- files |> map_dfr(read_csv)                        # map_dfr binds the results -- no separate ignore_index
                                                          # step needed, tibbles never carry a meaningful row index

# ── Pattern 2: Add a source column (track which file each row came from) ─────
df <- files |> map_dfr(function(f) {
  chunk <- read_csv(f)
  chunk$source_file <- path_ext_remove(path_file(f))    # e.g. '2024-01' -- fs::path_ext_remove/path_file = Path(f).stem
  chunk
})

# ── Pattern 3: Validate schema before combining ─────────────────────────────
# Problem: files from different periods may have different columns or types
dfs <- files |> map(read_csv)

# Check column consistency
expected_cols <- names(dfs[[1]])
for (i in seq_along(dfs)[-1]) {
  missing <- setdiff(expected_cols, names(dfs[[i]]))
  extra   <- setdiff(names(dfs[[i]]), expected_cols)
  if (length(missing) > 0 || length(extra) > 0) {
    cat(sprintf('File %d: missing=%s, extra=%s\n', i, paste(missing, collapse=","), paste(extra, collapse=",")))
  }
}

# Safe combine -- bind_rows() aligns on column names, fills missing with NA automatically
# (this IS the "outer join" behavior -- bind_rows() has no separate join= argument because
# aligning-and-filling is its only mode, unlike pandas' concat(join='outer'/'inner'))
df <- bind_rows(dfs)

# ── Pattern 4: Read all sheets from an Excel file ────────────────────────────
sheet_names <- readxl::excel_sheets("report.xlsx")
xl <- sheet_names |> set_names() |> map(~ readxl::read_excel("report.xlsx", sheet = .x))  # returns named list: list(sheet_name = df)
df <- bind_rows(xl)                                       # combine all sheets

# Or combine with sheet name as a column
df <- bind_rows(xl, .id = "sheet")                         # .id= adds the list name as a column directly -- more concise than pandas' manual loop

# ── Pattern 5: Large files -- read selectively ────────────────────────────────
# Only load the columns you need to avoid memory issues
needed_cols <- c('user_id', 'date', 'revenue')
df <- files |> map_dfr(~ read_csv(.x, col_select = all_of(needed_cols)))

**Common mistakes:**
- Assuming all files have the same schema — always validate before combining, especially with data from different time periods
- Loading all columns from all files when only a few are needed — use `col_select` to limit memory usage
- Using `bind_rows()` with mismatched types across files (e.g. `integer` in one file, `double` in another) — readr/dplyr will coerce, but check `glimpse()` after combining to verify the result is what you expect, same discipline as checking `df.dtypes` after `pd.concat`
- Forgetting that `bind_rows()` has no row-index concept to manage — unlike pandas' `pd.concat` (where `ignore_index=True` is an easy-to-forget but essential argument), tibbles simply don't carry duplicate row indices in the first place, so there is nothing to remember here

---
## Decision Guide

```
Need to change data shape?
+-- Wide -> Long (many columns -> one value column)  -> pivot_longer(cols, names_to, values_to)   (§1)
+-- Long -> Wide (categories -> separate columns)    -> pivot_wider(id_cols, names_from, values_from) (§1)
\-- Multi-value wide -> long                         -> pivot_longer(names_to = c(".value", ...))  (§1)

Need advanced group-by?
+-- Top-N per group                                  -> group_by() |> mutate(dense_rank()) + filter (§2)
+-- Remove groups failing a condition                -> group_by() |> filter(aggregate_condition)   (§2)
+-- Conditional sum within group (SUM CASE WHEN)     -> if_else() + group_by() |> summarise()        (§2)
+-- Group stats back on each row                     -> group_by() |> mutate()                       (§2)
\-- Custom aggregation (percentile, ratio)           -> summarise(stat = quantile(x, ...))           (§2)

Combining multiple tables?
+-- Row count grows after join                       -> aggregate right table first                  (§3)
+-- Rows in A with no match in B                      -> anti_join()                                  (§3)
+-- Many-to-many relationship                         -> aggregate one side or use a bridge table      (§3)
\-- Key type mismatch (rows drop to zero / error)    -> cast both keys to same type                  (§3)

Working with time series?
+-- Change frequency (daily -> monthly)               -> floor_date() |> group_by() |> summarise()    (§4)
+-- Fill gaps in time series                          -> build date spine + left_join                 (§4)
+-- Carry last known value forward                    -> fill(.direction = "down")                    (§4)
\-- Align different frequencies                      -> left_join + fill                              (§4)

Column contains lists / JSON?
\-- Flatten nested JSON                              -> jsonlite::fromJSON(flatten = TRUE)            (§5)

Column contains list-columns?
+-- One row per list element                          -> unnest(col)                                  (§5)
+-- Fixed-length list -> separate columns             -> purrr::map_dfr() + bind_cols()                (§5)
\-- Stringified JSON list                            -> jsonlite::fromJSON() (never eval/parse text)  (§5)

Reading many files?
+-- Same schema, just combine                         -> dir_ls() + map_dfr(read_csv)                 (§6)
+-- Track source file                                 -> add 'source_file' column per chunk            (§6)
+-- Schema may differ across files                    -> validate columns first, bind_rows() to align (§6)
\-- Multiple Excel sheets                            -> map over excel_sheets() + bind_rows(.id=)     (§6)
```